# Synthetic BoolQ via attractor perturbation

Companion to the paper's §*Attractor perturbations for data generation*. Injecting Gaussian noise into the output of a middle attention layer of Llama-3.1-8B-Instruct causes it, when prompted to generate BoolQ-shaped examples, to drift *around* a clean baseline rather than repeating it — producing diverse but on-topic synthetic (passage, question, answer) triples.

## Setup

In [ ]:
import os
import random
import re
import numpy as np
import torch
from tqdm.auto import tqdm
from rich import print as pprint
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

## Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct", torch_dtype=torch.bfloat16).to(device)
model = model.eval()

## Data

First run downloads BoolQ into `./boolq_cache/`; subsequent runs load from that local cache.

In [ ]:
ds = load_dataset("google/boolq", cache_dir="./boolq_cache")['train']

In [ ]:
prompts = []

for i, row in tqdm(enumerate(ds), total=len(ds)):
    prompts.append('passage: %s, question: %s, answer: %s' %(row['passage'], row['question'], row['answer']))
prompts = np.array(prompts)

keep_idx = random.sample(list(range(len(prompts))), 600)
prompts = prompts[keep_idx]

len(prompts)

## Attractor-perturbation hook

Register a single forward hook on the self-attention of layer 16 that adds a scaled Gaussian noise vector to the attention output. The hook reads `strength` and `noise` from the enclosing scope at call time, so the generation loop below can vary them per sample.

In [ ]:
target_layer = 16


def add_noise(target_layer):
    def hook(model, input, output):
        if abs(strength) > 1e-5:
            output = (output[0] + strength * noise.to(device), *output[1:])
        return output
    return hook


hooks = []
hooks.append(
    model.model.layers[target_layer].self_attn.register_forward_hook(add_noise(target_layer))
)


## Generation prompt

In [ ]:
target_prompt = 4

prompt = '''\n Now generate 3 different passages, questions, and answers similar to the example above. Please make sure each question you generate has a boolean answer that can be answered by the passage. Make sure each passage and question is different and sufficiently rephrased. Please make sure you generate passages, questions and both true and false answers.'''


## Noise sweep

One clean baseline (`strength=0`), then 10 samples at each of 5 noise strengths (`denom ∈ [10, 15, 20, 40, 100]` — smaller denominator = larger perturbation). 51 responses total.

In [ ]:
strength = 0
noise = torch.zeros(1, model.config.hidden_size).to(torch.bfloat16)

aug_prompt = prompts[target_prompt] + '\n' + prompt

responses = []

# Clean baseline (no perturbation).
torch.manual_seed(0)
with torch.no_grad():
    inputs = tokenizer(aug_prompt, return_tensors="pt").input_ids
    outputs = model.generate(
        inputs.to(device),
        max_new_tokens=500,
        do_sample=True, top_k=50, top_p=0.95, temperature=0.6,
        pad_token_id=tokenizer.eos_token_id, use_cache=True,
    ).detach().cpu()
output_text = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]
responses.append(output_text)
print('--- baseline (strength=0) ---')
pprint(output_text)

# Noise sweep: 10 samples at each of 5 noise strengths.
for denom in [10, 15, 20, 40, 100]:
    for _ in tqdm(range(10), desc=f'denom={denom}'):
        strength = 1
        torch.manual_seed(np.random.randint(low=0, high=1000000))
        noise = torch.randn(1, model.config.hidden_size).to(torch.bfloat16) / denom
        torch.manual_seed(0)
        with torch.no_grad():
            inputs = tokenizer(aug_prompt, return_tensors="pt").input_ids
            outputs = model.generate(
                inputs.to(device),
                max_new_tokens=500,
                do_sample=True, top_k=20, top_p=0.95, temperature=0.2,
                pad_token_id=tokenizer.eos_token_id, use_cache=True,
            ).detach().cpu()
        output_text = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]
        responses.append(output_text)
        print(f'--- denom={denom} ---')
        pprint(output_text)


## Post-processing

In [ ]:
for i in range(len(responses)):
    responses[i] = re.sub(r'\s+', ' ', responses[i].strip())

In [ ]:
def split_boolq_sample(sample):

    sample = re.sub(r'(?i)\bp\s*\d*\s*:', 'Passage:', sample)
    sample = re.sub(r'(?i)\bq\s*\d*\s*:', 'Question:', sample)
    sample = re.sub(r'(?i)\ba\s*\d*\s*:', 'Answer:', sample)

    splits = re.split(r'(?i)\bpassage\s*\d*\s*:', sample)

    for i in range(len(splits)):
        splits[i] = re.split(r'(?i)\bquestion\s*\d*\s*:', splits[i])[:2]
    for i in range(len(splits)):
        if (len(splits[i]) > 1):
            splits[i][1] = re.split(r'(?i)\banswer\s*\d*\s*:', splits[i][1])[:2]


    passages = []
    questions = []
    answers = []

    for s in splits:
        if (len(s) != 2 or len(s[1]) != 2):
            continue 

        passages.append(s[0].strip())
        questions.append(s[1][0].strip())


        answer = s[1][1].strip()
        if ('True' in answer or 'true' in answer or 'Yes' in answer or 'yes' in answer):
            answers.append(True)
        else:
            answers.append(False)
    

    return passages, questions, answers
        

## Example output

In [ ]:
# Parse every response and summarise how many (passage, question, answer) triples it yielded.
for i, response in enumerate(responses):
    passages, questions, answers = split_boolq_sample(response)
    print(f'response {i:>3}: {len(passages)} parsed triples')

print()
print('--- detail: response[1] ---')
passages, questions, answers = split_boolq_sample(responses[1])
for p, q, a in zip(passages, questions, answers):
    pprint({'passage': p, 'question': q, 'answer': a})
